# P9a: PyTorch asoslari

Bu notebook bugungi deep learning practice uchun qisqa tayyorgarlik. Maqsad PyTorch'ni alohida, kichik misollarda ko'rish: tensor, autograd, `nn.Module`, `Dataset/DataLoader`, training loop va LSTM shape'lari.

Bu notebook `d10_p9_textgen`dan oldin ishlatilishi mumkin. Tashqi dataset kerak emas.


## 1. Muhit

Kaggle'da PyTorch odatda tayyor o'rnatilgan bo'ladi. GPU shart emas: bu notebookdagi misollar CPU'da ham tez ishlaydi.


In [ ]:
import os
import sys
import random
import math
import tempfile
from collections import Counter

os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "matplotlib"))

import numpy as np

random.seed(42)
np.random.seed(42)

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    torch.manual_seed(42)
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    torch = None
    nn = None
    Dataset = object
    DataLoader = None

HAS_MATPLOTLIB = None

print("Python:", sys.version.split()[0])
print("NumPy :", np.__version__)
print("Torch :", torch.__version__ if HAS_TORCH else "topilmadi")
print("Device:", "cuda" if HAS_TORCH and torch.cuda.is_available() else "cpu")


## 2. Tensor va shape

PyTorch tensorlari NumPy arraylariga o'xshaydi, lekin GPU va autograd bilan ishlaydi. Deep learningdagi ko'p xatolar shape noto'g'ri tushunilganidan keladi.


In [ ]:
if HAS_TORCH:
    x = torch.tensor([[1.0, 2.0, 3.0],
                      [4.0, 5.0, 6.0]])
    weights = torch.tensor([[0.2, -0.3],
                            [0.5,  0.1],
                            [0.7,  0.4]])

    # SIZNING KODINGIZ:
    # 1. matrix multiplication qiling: x @ weights
    # 2. softmax ni dim=1 bo'yicha hisoblang
    logits = ...
    probabilities = ...

    print("logits shape:", tuple(logits.shape))
    print("probabilities:\n", probabilities)
else:
    print("Kaggle'da torch mavjud bo'ladi. Bu local muhitda torch yo'q.")


## 3. Autograd

`requires_grad=True` bo'lsa, PyTorch amallar grafigini eslab qoladi. `loss.backward()` chaqirilganda gradientlar `.grad` ichida paydo bo'ladi.


In [ ]:
if HAS_TORCH:
    w = torch.tensor(0.0, requires_grad=True)
    target = torch.tensor(3.0)

    # SIZNING KODINGIZ:
    # loss = (w - target) ** 2
    # loss.backward()
    loss = ...
    ...

    print("loss:", float(loss))
    print("d loss / d w:", float(w.grad))
else:
    print("Autograd demo uchun torch kerak.")


## 4. Gradient descent qo'lda

Optimizer nima qilayotganini ko'rish uchun avval bitta parametrni qo'lda yangilaymiz.


In [ ]:
if HAS_TORCH:
    w = torch.tensor(0.0, requires_grad=True)
    lr = 0.2
    history = []

    for step in range(15):
        # SIZNING KODINGIZ:
        # 1. loss hisoblang
        # 2. backward qiling
        # 3. torch.no_grad() ichida w -= lr * w.grad
        # 4. gradientni zero qiling
        loss = ...
        ...
        history.append(float(loss))

    print("w:", round(float(w), 3))
else:
    history = []
    print("Manual training demo uchun torch kerak.")


In [ ]:
if history:
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(5, 3))
        plt.plot(history, marker="o")
        plt.title("Loss kamayishi")
        plt.xlabel("step")
        plt.ylabel("loss")
        plt.grid(True, alpha=0.3)
        plt.show()
    except ImportError:
        print("Loss history:", history)
else:
    print("Plot uchun training history yo'q.")


## 5. Kichik NLP dataset

Endi juda kichik sentiment dataset bilan `nn.Module` va training loop ko'ramiz. Bu katta model emas; maqsad PyTorch mexanikasini tushunish.


In [ ]:
texts = [
    "mahsulot juda yaxshi",
    "xizmat ajoyib va tez",
    "taom mazali ekan",
    "telefon menga yoqdi",
    "yetkazish yomon bo'ldi",
    "mahsulot sifatsiz ekan",
    "xizmat juda sekin",
    "narxi qimmat va yomon",
]
labels = [1, 1, 1, 1, 0, 0, 0, 0]

vocab = sorted({token for text in texts for token in text.split()})
word_to_index = {word: index for index, word in enumerate(vocab)}

print("Vocab size:", len(vocab))
print(word_to_index)


In [ ]:
def vectorize(text):
    vector = np.zeros(len(vocab), dtype=np.float32)
    for token in text.split():
        if token in word_to_index:
            vector[word_to_index[token]] += 1.0
    return vector

X_np = np.vstack([vectorize(text) for text in texts])
y_np = np.array(labels, dtype=np.int64)

print("X shape:", X_np.shape)
print("y shape:", y_np.shape)
print("Birinchi matn vectori:", X_np[0])


## 6. `nn.Module`

PyTorch model klassi odatda ikki qismdan iborat:

- `__init__`: layerlarni yaratadi;
- `forward`: inputdan outputgacha hisoblash yo'lini belgilaydi.


In [ ]:
if HAS_TORCH:
    class TinySentimentNet(nn.Module):
        def __init__(self, input_dim, hidden_dim=8):
            super().__init__()
            # SIZNING KODINGIZ:
            # Linear -> ReLU -> Linear arxitekturasini yozing.
            self.layers = ...

        def forward(self, features):
            return self.layers(features)

    model = TinySentimentNet(input_dim=len(vocab))
    print(model)
else:
    TinySentimentNet = None
    model = None
    print("nn.Module demo uchun torch kerak.")


## 7. Dataset va DataLoader

`Dataset` bitta namunani qaytaradi. `DataLoader` esa batch qiladi, shuffle qiladi va training loopni tartibli qiladi.


In [ ]:
if HAS_TORCH:
    class SentimentDataset(Dataset):
        def __init__(self, X, y):
            # SIZNING KODINGIZ: X va y ni torch tensorlarga aylantiring.
            self.X = ...
            self.y = ...

        def __len__(self):
            return len(self.y)

        def __getitem__(self, index):
            return self.X[index], self.y[index]

    dataset = SentimentDataset(X_np, y_np)
    loader = DataLoader(dataset, batch_size=4, shuffle=True)
    batch_X, batch_y = next(iter(loader))
    print("batch_X:", tuple(batch_X.shape))
    print("batch_y:", tuple(batch_y.shape), batch_y.tolist())
else:
    dataset = None
    loader = None
    print("Dataset/DataLoader demo uchun torch kerak.")


## 8. Training loop

Ko'p PyTorch kodlarining yuragi shu tartib:

1. `optimizer.zero_grad()`
2. `logits = model(batch_X)`
3. `loss = loss_fn(logits, batch_y)`
4. `loss.backward()`
5. `optimizer.step()`


In [ ]:
if HAS_TORCH:
    model = TinySentimentNet(input_dim=len(vocab))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
    loss_fn = nn.CrossEntropyLoss()
    epoch_losses = []

    for epoch in range(40):
        total_loss = 0.0
        for batch_X, batch_y in loader:
            # SIZNING KODINGIZ:
            # zero_grad -> forward -> loss -> backward -> step
            ...
            total_loss += float(loss)
        epoch_losses.append(total_loss / len(loader))

    print("final loss:", round(epoch_losses[-1], 4))
else:
    epoch_losses = []
    print("Training loop uchun torch kerak.")


In [ ]:
if epoch_losses:
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(5, 3))
        plt.plot(epoch_losses)
        plt.title("Tiny classifier training loss")
        plt.xlabel("epoch")
        plt.ylabel("loss")
        plt.grid(True, alpha=0.3)
        plt.show()
    except ImportError:
        print(epoch_losses[:5], "...", epoch_losses[-5:])
else:
    print("Plot uchun epoch_losses yo'q.")


## 9. Inference

Trainingdan keyin prediction paytida gradient kerak emas. Shuning uchun `model.eval()` va `torch.no_grad()` ishlatiladi.


In [ ]:
if HAS_TORCH:
    def predict_sentiment(text):
        model.eval()
        features = torch.tensor(vectorize(text), dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            # SIZNING KODINGIZ: logits, softmax, argmax
            logits = ...
            probabilities = ...
            prediction = ...
        label_name = "ijobiy" if prediction == 1 else "salbiy"
        return label_name, probabilities.tolist()

    for sample in ["mahsulot yaxshi", "xizmat yomon", "taom juda mazali"]:
        label_name, probabilities = predict_sentiment(sample)
        print(sample, "->", label_name, [round(p, 3) for p in probabilities])
else:
    print("Prediction demo uchun torch kerak.")


## 10. Sequence model shape'lari

Text generation practice'da input odatda shunday bo'ladi:

`token_ids -> embedding -> LSTM -> output layer`

Quyidagi cell shape'larni ko'rsatadi. Bu LSTM ichki matematikasini emas, PyTorchdagi obyektlar qanday ulanganini ko'rsatadi.


In [ ]:
if HAS_TORCH:
    batch_size = 2
    seq_len = 5
    vocab_size = 20
    embed_dim = 8
    hidden_size = 16

    token_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
    embedding = nn.Embedding(vocab_size, embed_dim)
    lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)

    embedded = embedding(token_ids)
    outputs, (hidden_state, cell_state) = lstm(embedded)

    print("token_ids   :", tuple(token_ids.shape))
    print("embedded    :", tuple(embedded.shape))
    print("outputs     :", tuple(outputs.shape))
    print("hidden_state:", tuple(hidden_state.shape))
    print("cell_state  :", tuple(cell_state.shape))
else:
    print("Sequence shape demo uchun torch kerak.")


## 11. Bugungi practice bilan bog'lash

Endi `d10_p9_textgen` notebookida ko'radigan char-LSTM kodi shunchaki shu PyTorch qismlarining ketma-ketligi bo'ladi:

- character IDlar;
- `nn.Embedding`;
- `nn.LSTM`;
- `nn.Linear` output;
- `CrossEntropyLoss`;
- temperatura sampling.
